In [84]:
import pandas as pd
import numpy as np
from pandas.io.clipboard import paste
df = pd.read_csv(
    '../data/raw/ad_product_to_content_taxonomy_mapping.tsv',
    sep='\t',
    dtype={
        'Unique ID': 'string',
        'Unique ID 2': 'string',
        'Name': 'string',
        'Tier 1': 'string',
        'Tier 2': 'string',
        'Tier 3': 'string'
    },
    engine='python',  # More forgiving than C engine
    on_bad_lines='warn',  # Warn about bad lines but continue
    encoding='utf-8',
    quoting=3  # QUOTE_NONE
)

In [89]:
df = pd.read_csv(
    '../data/raw/ad_product_to_content_taxonomy_mapping.tsv',
    sep='\t',
    dtype={
        'Unique ID': 'string',
        'Unique ID 2': 'string',
        'Namesss': 'string',
        'Tier 1': 'string',
        'Tier 2': 'string',
        'Tier 3': 'string'
    },
    engine='python',  # More forgiving than C engine
    on_bad_lines='warn',  # Warn about bad lines but continue
    encoding='utf-8',
    quoting=3  # QUOTE_NONE
)
df = df.rename(columns={'Unique ID': 'ad_taxonomy_id', 'Unique ID 2': 'content_taxonomy_id'})
ad_product_to_content_mapping = pd.Series(df['ad_taxonomy_id'].values, index=df['content_taxonomy_id']).to_dict()


In [90]:
ad_product_to_content_mapping

{'v9i3On': '1582',
 'Rm3SiT': '1001',
 '211': '1007',
 '150': '1509',
 '201': '1008',
 '155': '1009',
 '53': '1048',
 '91': '1011',
 '55': '1016',
 '101': '1019',
 '121': '1038',
 '78': '1370',
 '119': '1043',
 '552': '1498',
 '575': '1060',
 '566': '1067',
 '582': '1062',
 '570': '1065',
 '165': '1066',
 '561': '1074',
 '573': '1071',
 '240': '1076',
 '263': '1081',
 '261': '1078',
 '239': '1504',
 '602': '1440',
 '596': '1467',
 '323': '1094',
 '553': '1275',
 '554': '1277',
 '594': '1276',
 '556': '1093',
 '632': '1472',
 '633': '1103',
 '630': '1107',
 '617': '1108',
 '636': '1118',
 '634': '1112',
 '635': '1496',
 '600': '1117',
 '640': '1470',
 '680': '1122',
 '210': '1501',
 '192': '1126',
 '132': '1314',
 '161': '1147',
 '479': '1151',
 '473': '1507',
 '274': '1502',
 '123': '1311',
 '286': '1208',
 '590': '1219',
 '591': '1220',
 '592': '1218',
 '422': '1505',
 '431': '1228',
 '269': '1251',
 '234': '1253',
 '223': '1395',
 '231': '1266',
 '280': '1404',
 '188': '1261',
 '255'

In [80]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1180 entries, 0 to 1179
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Unique ID  1179 non-null   string
 1   Parent ID  1144 non-null   string
 2   Name       1179 non-null   string
 3   Tier 1     1179 non-null   string
 4   Tier 2     1144 non-null   string
 5   Tier 3     584 non-null    string
 6   Tier 4     60 non-null     object
dtypes: object(1), string(6)
memory usage: 64.7+ KB


In [81]:
import pandas as pd

def build_nested_taxonomy_json(df):
    df = df.rename(columns={'Unique ID': 'id_str', 'Parent ID': 'parent_id_str'})

    valid_id_mask = df['id_str'].notna() & (df['id_str'] != '')
    df = df[valid_id_mask].copy()

    name_mapping = pd.Series(df['Name'].values, index=df['id_str']).to_dict()

    df.loc[df['parent_id_str'].isna(), 'parent_id_str'] = None

    graph = {id_str: [] for id_str in df['id_str']}
    all_children = set()

    for row in df.itertuples(index=False):
        id_str = row.id_str
        parent_id_str = row.parent_id_str

        if pd.notna(parent_id_str) and parent_id_str != id_str:
            if parent_id_str in graph:
                graph[parent_id_str].append(id_str)
            else:
                graph[parent_id_str] = [id_str]
            all_children.add(id_str)

    all_nodes = set(graph.keys())
    root_nodes = all_nodes - all_children

    def dfs(node_id, tier):
        new_node = {
            "id": node_id,
            "name": name_mapping.get(node_id, "Unknown"),
            "tier": tier
        }

        children_ids = graph[node_id]

        if children_ids:
            child_list = []
            for child_id in children_ids:
                child_node = dfs(child_id, tier + 1)
                child_list.append(child_node)

            new_node["children"] = child_list

        return new_node

    taxonomy_tree = []
    for root in root_nodes:
        taxonomy_tree.append(dfs(root, 1))

    return taxonomy_tree


result = build_nested_taxonomy_json(df)

In [82]:
result

[{'id': '123',
  'name': 'Careers',
  'tier': 1,
  'children': [{'id': '124', 'name': 'Apprenticeships', 'tier': 2},
   {'id': '125', 'name': 'Career Advice', 'tier': 2},
   {'id': '126', 'name': 'Career Planning', 'tier': 2},
   {'id': '127',
    'name': 'Job Search',
    'tier': 2,
    'children': [{'id': '128', 'name': 'Job Fairs', 'tier': 3},
     {'id': '129', 'name': 'Resume Writing and Advice', 'tier': 3}]},
   {'id': '130', 'name': 'Remote Working', 'tier': 2},
   {'id': '131', 'name': 'Vocational Training', 'tier': 2}]},
 {'id': '391',
  'name': 'Personal Finance',
  'tier': 1,
  'children': [{'id': '392', 'name': 'Consumer Banking', 'tier': 2},
   {'id': '393',
    'name': 'Financial Assistance',
    'tier': 2,
    'children': [{'id': '394',
      'name': 'Government Support and Welfare',
      'tier': 3},
     {'id': '395', 'name': 'Student Financial Aid', 'tier': 3}]},
   {'id': '396', 'name': 'Financial Planning', 'tier': 2},
   {'id': '397', 'name': 'Frugal Living', 'tier